In [5]:
COLORLEFT = '#31A2AC'
COLORRIGHT = '#FF8D3F'

# Load modules and data
# from linares_plot import *

import json
from scipy import special
#Import all needed libraries

import os
import pandas as pd
import numpy as np


# os.getcwd() 
# os.chdir('C:/Users/Tiffany/Google Drive/WORKING_MEMORY/EXPERIMENTS/ANALYSIS CODES')
# import tools_code as tools
from scipy.optimize import curve_fit

def exp_decay(t, N0, tau):
    return N0 * np.exp(-t / tau)

In [6]:
def fix_delays_model(row):
    delays=[0.0,1000,3000,10000,15000,20000,25000,27500,30000]
    # delays= [0.0,10000,15000,20000,25000,30000, 35000,40000,50000,60000]
    # delays=[100,1000,3000,10000]

    return delays[row['idelays']-1]/1000

def fix_delays(row):
    # delays=[0.0,100,1000,3000,10000,15000,20000,25000,30000]
    delays=[100,1000,3000,10000]
    return delays[row['idelays']-1]/1000

def streak(row):
    value = 0
    '''
    Skipping the first trial, look for whether there was a sequences of same side trials or opposite side trials
    '''
    if row['trial'] != 1:
        if row['stim'] == row['past_choices'][0]:
            value = 1
            if row['past_choices'][0] == row['past_choices'][1]:
                value = 2
                if row['past_choices'][1] == row['past_choices'][2]:
                    value = 3
        else: 
            value = -1
            if row['past_choices'][0] == row['past_choices'][1]:
                value = -2
                if row['past_choices'][1] == row['past_choices'][2]:
                    value = -3 
                else:
                    value = -2
                    return value
            else:
                value = -1
                return value
    return value

# Create a repeating vector
def repeat_choice(row):
    val = 0
    if row['choices'] == row['past_choices'][0]:
        val = 1
    elif row['past_choices'][0] == 0:
        val = np.nan
    else:
        val = 0
    return val

def repeat_choice_side(row):
    '''
    Parameters
    ----------
    row : look row by row whether there was a repetition of LEFT response (1) or RIGHT response (2)

    Returns
    -------
    TYPE
        DESCRIPTION.

    '''
    # Compare the current response with the previous one. If that matches, return a 1 meaning it repeated. 
    if row['trial'] != 0:
        # Compare the current response with the previous one. .  
        if row['choices'] == row['past_choices'][0]:
            # if it matched and the answer was 1, it means that it repeated a right response
            if row['choices'] == 1:
                return 2
            # if it matched and the answer was 0, it means that it repeated a left response
            else:
                return 1
        elif row['past_choices'][0] == 0:
            return 0
        else:
            return 0
    else:
        return np.nan
        
def compute_window(data, runningwindow,option):
    """
    Computes a rolling average with a length of runningwindow samples.
    """
    performance = []
    end=False
    for i in range(len(data)):
        if data['trial'].iloc[i] <= runningwindow:
            # Store the first index of that session
            if end == False:
                start=i
                end=True
            performance.append(round(np.mean(data[option].iloc[start:i + 1]), 2))
        else:
            end=False
            performance.append(round(np.mean(data[option].iloc[i - runningwindow:i]), 2))
    return performance

def compute_window_centered(data, runningwindow,option):
    """
    Computes a rolling average with a length of runningwindow samples.
    """
    performance = []
    start_on=False
    for i in range(len(data)):
        if data['trial'].iloc[i] <= int(runningwindow/2):
            # Store the first index of that session for the first initial trials
            if start_on == False:
                start=i
                start_on=True
            performance.append(round(np.mean(data[option].iloc[start:i + int(runningwindow/2)]), 2))
        elif i < (len(data)-runningwindow):
            if data['trial'].iloc[i] > data['trial'].iloc[i+runningwindow]:
                # Store the last values for the end of the session
                if end == True:
                    end_value = i+runningwindow-1
                    end = False
                performance.append(round(np.mean(data[option].iloc[i:end_value]), 2))
                
            else: # Rest of the session
                start_on=False
                end = True
                performance.append(round(np.mean(data[option].iloc[i - int(runningwindow/2):i+int(runningwindow/2)]), 2))
            
        else:
            performance.append(round(np.mean(data[option].iloc[i:len(data)]), 2))
    return performance

In [17]:
    file_path = r'E:\HMM_mice\real\{}_behavior.json'.format(animal)

    # Step 1: load the outer string
    with open(file_path, 'r') as j:
        outer_string = json.load(j)  # this is still a string

    df_real = pd.DataFrame(outer_string)

In [38]:
df_real

,choices,Pstate,stim,past_choices,LL,past_rewards,day,idelays
0,"[2, 2, 2, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, ...","[[0.9823885638, 0.0176114362], [0.9427182999, ...","[2, 2, 2, 1, 2, 1, 2, 2, 1, 2, 2, 2, 2, 1, 2, ...","[[-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0,...","[2021-12-02, 2021-12-02, 2021-12-02, 2021-12-0...","[3, 3, 2, 2, 1, 2, 4, 1, 4, 4, 1, 2, 3, 1, 3, ..."
1,"[2, 1, 2, 1, 2, 2, 2, 1, 1, 2, 1, 1, 2, 2, 1, ...","[[0.8666020364, 0.1333979636], [0.9974246575, ...","[2, 1, 2, 1, 2, 2, 2, 1, 1, 2, 1, 1, 2, 2, 1, ...","[[1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-03, 2021-12-03, 2021-12-03, 2021-12-0...","[2, 1, 4, 2, 2, 2, 2, 3, 2, 2, 1, 3, 4, 4, 1, ..."
2,"[1, 2, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, ...","[[0.9808173136, 0.0191826864], [0.8963526967, ...","[1, 1, 2, 1, 1, 1, 2, 2, 1, 2, 2, 2, 2, 2, 2, ...","[[1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1...",3490.459153,"[[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-04, 2021-12-04, 2021-12-04, 2021-12-0...","[3, 4, 1, 1, 1, 1, 2, 3, 1, 2, 2, 2, 2, 4, 2, ..."
3,"[1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 1, 2, 1, ...","[[0.8788149286, 0.1211850714], [0.9918157549, ...","[1, 2, 2, 2, 2, 2, 2, 1, 2, 1, 2, 1, 1, 2, 1, ...","[[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, ...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-06, 2021-12-06, 2021-12-06, 2021-12-0...","[3, 2, 2, 3, 2, 4, 3, 1, 2, 4, 4, 4, 1, 1, 4, ..."
4,"[2, 2, 1, 2, 2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, ...","[[0.7703923915, 0.2296076085], [0.874833148, 0...","[2, 2, 1, 2, 2, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, ...","[[1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0,...","[2021-12-07, 2021-12-07, 2021-12-07, 2021-12-0...","[4, 1, 1, 1, 3, 4, 1, 3, 2, 2, 4, 1, 3, 4, 3, ..."
5,"[1, 2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, ...","[[0.8602578009, 0.1397421991], [0.9990393264, ...","[1, 2, 1, 2, 1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, ...","[[-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-09, 2021-12-09, 2021-12-09, 2021-12-0...","[1, 2, 1, 1, 4, 4, 2, 1, 3, 4, 2, 4, 1, 4, 4, ..."
6,"[2, 1, 1, 2, 1, 1, 2, 1, 2, 1, 2, 2, 1, 2, 2, ...","[[0.9668311534, 0.0331688466], [0.9859443702, ...","[2, 1, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 2, ...","[[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0,...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0,...","[2021-12-11, 2021-12-11, 2021-12-11, 2021-12-1...","[1, 2, 3, 2, 3, 2, 1, 3, 4, 1, 4, 4, 3, 4, 4, ..."
7,"[2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, ...","[[0.8830056184, 0.1169943816], [0.7348737742, ...","[2, 2, 2, 1, 1, 2, 1, 2, 2, 2, 2, 2, 2, 1, 2, ...","[[-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0,...",3490.459153,"[[1.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-13, 2021-12-13, 2021-12-13, 2021-12-1...","[1, 2, 2, 2, 4, 2, 4, 4, 3, 4, 3, 3, 1, 2, 1, ..."
8,"[2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 1, 2, 2, ...","[[0.9693464705, 0.0306535295], [0.9892275279, ...","[2, 1, 2, 2, 2, 1, 2, 2, 1, 2, 2, 2, 1, 2, 2, ...","[[-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0,...","[2021-12-15, 2021-12-15, 2021-12-15, 2021-12-1...","[2, 2, 4, 1, 4, 4, 1, 2, 2, 3, 2, 2, 2, 1, 4, ..."
9,"[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 1, ...","[[0.9987333552, 0.0012666448], [0.9770462208, ...","[2, 1, 2, 2, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 1, ...","[[-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0,...",3490.459153,"[[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,...","[2021-12-17, 2021-12-17, 2021-12-17, 2021-12-1...","[1, 2, 3, 4, 1, 2, 3, 4, 4, 4, 3, 2, 4, 4, 2, ..."


In [ ]:
concat_df = pd.DataFrame()
for i, row in df_real.iterrows():
    new_df = pd.DataFrame()
    new_df['choices'] = row['choices']
    new_df['Pstate'] = row['Pstate']
    new_df['stim'] = row['stim']
    new_df['past_choices'] = row['past_choices']
    new_df['LL'] = row['LL']
    new_df['past_rewards'] = row['past_rewards']
    new_df['day'] = row['day']
    new_df['idelays'] = row['idelays']
    new_df['trial'] = range(1, len(row['choices']) + 1)
    concat_df = pd.concat([concat_df, new_df], ignore_index=True)
    
# First element of each list
concat_df["state"] = concat_df["Pstate"].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else None)


In [ ]:
# First element of each list
concat_df["state"] = concat_df["Pstate"].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else None)


In [52]:
concat_df

,choices,Pstate,stim,past_choices,LL,past_rewards,day,idelays,trial,a_first
0,2,"[0.9823885638, 0.0176114362]",2,"[-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0,...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, ...",2021-12-02,3,1,0.982389
1,2,"[0.9427182999, 0.0572817001]",2,"[1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0,...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, ...",2021-12-02,3,2,0.942718
2,2,"[0.9406240277, 0.0593759723]",2,"[1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, ...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, ...",2021-12-02,2,3,0.940624
3,1,"[0.9982707822, 0.0017292178]",1,"[1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",2021-12-02,2,4,0.998271
4,2,"[0.9852894207, 0.0147105793]",2,"[-1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...",2021-12-02,1,5,0.985289
...,...,...,...,...,...,...,...,...,...,...
8597,2,"[0.9973461812, 0.0026538188]",2,"[-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, ...",2022-01-22,1,133,0.997346
8598,2,"[0.8465746563, 0.1534253437]",2,"[1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, ...",2022-01-22,2,134,0.846575
8599,2,"[0.4937963054, 0.5062036946]",1,"[1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1...",3490.459153,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, ...",2022-01-22,3,135,0.493796
8600,1,"[0.6840952828, 0.3159047172]",1,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3490.459153,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2022-01-22,4,136,0.684095


In [53]:
def create_data(animal,extra):
    os.getcwd() 
    file_path = r'E:\HMM_mice\real\{}_behavior.json'.format(animal)

    # Step 1: load the outer string
    with open(file_path, 'r') as j:
        outer_string = json.load(j)  # this is still a string

    df_real = pd.DataFrame(outer_string)

    new_df_real = pd.DataFrame()
    for i, row in df_real.iterrows():
        new_df = pd.DataFrame()
        new_df['choices'] = row['choices']
        new_df['Pstate'] = row['Pstate']
        new_df['stim'] = row['stim']
        new_df['past_choices'] = row['past_choices']
        new_df['LL'] = row['LL']
        new_df['past_rewards'] = row['past_rewards']
        new_df['day'] = row['day']
        new_df['idelays'] = row['idelays']
        new_df['trial'] = range(1, len(row['choices']) + 1)
        new_df_real = pd.concat([new_df_real, new_df], ignore_index=True)
        
    # First element of each list
    new_df_real["WM"] = new_df_real["Pstate"].apply(lambda x: x[0] if isinstance(x, (list, tuple)) and len(x) > 0 else None)
    new_df_real["RL"] = new_df_real["Pstate"].apply(lambda x: x[1] if isinstance(x, (list, tuple)) and len(x) > 1 else None)

    # ---------------
    def after_correct(row):
        return row['past_rewards'][0]

    new_df_real['after_correct'] = new_df_real.apply(after_correct, axis=1)

    # ---------------
    
    # Next delays
    new_df_real['delays'] = new_df_real.apply(fix_delays, axis=1)

    # -------

    new_df_real.rename(columns = {'choices_index':'trial'}, inplace = True)
    new_df_real['session'] = new_df_real['day']

    # Revalue variables stim, choices and create hit vector. -1 means Left and 1 means right. 1 for correct in hit and 0 for incorrect
    new_df_real['stim']= np.where(new_df_real['stim'] == 1, 0, 1)
    new_df_real['choices']= np.where(new_df_real['choices'] == 1, 0, 1)
    new_df_real['hit'] = np.where(new_df_real['stim'] == new_df_real['choices'], 1, 0)

    new_df_real['repeat'] = new_df_real.apply(repeat_choice,axis=1)

    new_df_real['repeat_choice_side'] = new_df_real.apply(repeat_choice_side,axis=1)
    
    # Correct the sequences of repetitions and alternations
    new_df_real['streak'] = new_df_real.apply(streak,axis=1)

    # ------
#     print('Total trials for WM: ', new_df_real[new_df_real["WM"]>0.75]["stim"].count()/new_df_real["stim"].count())
#     print('Total trials for RL: ', new_df_real[new_df_real["RL"]>0.25]["stim"].count()/new_df_real["stim"].count())

#     print('Total trials for WM: ', new_df[new_df["state"]==1]["stim"].count()/new_df["stim"].count())
#     print('Total trials for RL: ', new_df[new_df["state"]==0]["stim"].count()/new_df["stim"].count())
    # ---------

    new_df_real.drop(columns = {"idelays","Pstate"}, inplace=True)
    
    return new_df_real

In [56]:
full_fit = pd.DataFrame(dtype = 'float')
all_data = pd.DataFrame(dtype = 'float')
all_data_model = pd.DataFrame(dtype = 'float')
df_correlation = pd.DataFrame(dtype = 'float')
new_df_real = pd.DataFrame(dtype = 'float')

# extra='all'
# list_name=["E03", "E04", "E05_3", "E05_10", "E06", "E07_3", "E07_10", "E08" ,"E09", "E10", "E11", "E12_3", "E12_10", "E13" ,"E14", "E15_3", "E15_10", "E16_3", "E16_10", "E17_3", "E17_10", "E18" ,"E19", "E20_3", "E20_10", "E21", "E22", "N02", "N03", "N04", "N05_3", "N05_10", "N07_3", "N07_10", "N08", "N09", "N11_3", "N11_10", "N13", "N19", "N20", "N21", "N22", "N24_3", "N24_10", "N25_3", "N25_10", "N26", "N27_3", "N27_10", "N28_3", "N28_10", "C10b", "C12", "C13", "C15", "C18", "C19", "C20", "C22", "C28_3", "C28_10",  "C32", "C34", "C36", "C37_3", "C37_10", "C38", "C39" ]

# list_name=["E03", "E04", "E05_3", "E05_10", "E06", "E07_3", "E07_10", "E08" ,"E09", "E10", "E11", "E12_3", "E12_10", "E13" ,"E14", "E15_3", "E15_10", "E16_3", "E16_10", "E17_3", "E17_10", "E18" ,"E19", "E20_3", "E20_10", "E21", "E22",
# "N02", "N03", "N04", "N05_3", "N05_10", "N07_3", "N07_10", "N08", "N09", "N11_3", "N11_10", "N13", "N19", "N20", "N21", "N22", "N24_3", "N24_10", "N25_3", "N25_10", "N26", "N27_3", "N27_10", "N28_3", "N28_10",
# "C10b", "C12", "C13", "C15", "C18", "C19", "C20", "C22", "C28_3", "C28_10",  "C32", "C34", "C36", "C37_3", "C37_10", "C38", "C39"]

# list_name = ["E05_10"]
# list_name=["C12_3", "C13_10", "C10b_3", "C10b_10", "C15_3", "C18_3", "C18_10", "C19_3", "C19_10", "C20_3", "C22_3", "N08_3",
#            "N08_10", "N13_3", "N07_3", "N07_10", "N09_3", "N11_3", "N11_10", "N03_3", "N03_10", "N04_3", "N04_10", "N02_3", 
#            "N02_10","N05_3", "N05_10", "C28_3", "C28_10", "C32_3",  "C34_3", "C34_10", "C39_10", "C37_3", "C37_10", "C38_10", "C36_3",
#            "C36_10", "N19_3", "N19_10", "N27_3", "N27_10", "N24_3", "N24_10", "N20_10", "N22_3", "N22_10", "N28_3", "N28_10", "N26_3",
#            "N26_10", "N21_3", "N25_3", "N25_10", "E04_10", "E08_3", "E08_10", "E10_3","E10_10", "E05_3", "E05_10", "E03_10", "E07_3", 
#            "E07_10","E06_3", "E11_10", "E13_10", "E14_3", "E14_10", "E09_3", "E09_10", "E12_3", "E12_10", "E21_3", "E21_10", "E22_3", 
#            "E22_10","E16_10", "E20_3", "E20_10", "E17_10", "E19_10", "E18_3", "E18_10", "E15_3", "E15_10"]

# With more than 1000 trials
# list_name=["C12_3", "C13_10", "C10b_3", "C10b_10", "C15_3", "C18_3", "C19_3", "C19_10", "C20_3", "C22_3", "N08_3",
#            "N08_10", "N13_3", "N07_3", "N07_10", "N09_3", "N11_3", "N11_10", "N03_3", "N03_10", "N04_3", "N04_10", "N02_3", 
#            "N02_10","N05_3", "N05_10", "C28_3", "C28_10", "C32_3",  "C34_3", "C34_10", "C39_10", "C37_3", "C38_10", "C36_3",
#            "C36_10", "N19_3", "N19_10", "N27_3", "N27_10", "N24_3", "N24_10", "N20_10", "N22_3", "N22_10", "N28_3", "N28_10", "N26_3",
#            "N26_10", "N21_3","E04_10", "E08_3", "E08_10", "E10_3","E10_10", "E05_3", "E05_10", "E03_10", "E07_3", 
#            "E07_10","E06_3", "E11_10", "E13_10", "E14_10", "E09_10", "E12_3", "E12_10", "E21_3", "E21_10", "E22_3", 
#            "E22_10","E16_10", "E20_3", "E20_10", "E17_10", "E19_10", "E18_3", "E18_10", "E15_3", "E15_10"]

list_name=["E04_10", "E14_10","E11_10", "E20_3", "E20_10","E22_3", "E22_10",
             "E13_10", "E17_3", "E17_10", "E19_10" ]

for extra in ["all"]:

    for animal in list_name:
        print(animal)
        if extra == 'all':
            new_df_real = create_data(animal,extra)
        
        if extra == 'all':
            new_df_real['subject'] = animal
            new_df_real['model'] = extra
            
            if animal[-1] =='3':
                new_df_real['animal_delay'] = 3   
            else:
                new_df_real['animal_delay'] = 10  
                
            all_data = pd.concat([new_df_real,all_data])

E04_10
E14_10
E11_10
E20_3
E20_10
E22_3
E22_10
E13_10
E17_3
E17_10
E19_10


In [62]:
all_data.groupby('subject').day.unique().iloc[9]

array(['2021-12-02', '2021-12-03', '2021-12-04', '2021-12-06',
       '2021-12-07', '2021-12-09', '2021-12-11', '2021-12-13',
       '2021-12-15', '2021-12-17', '2021-12-24', '2021-12-29',
       '2022-01-10', '2022-01-11', '2022-01-12', '2022-01-13',
       '2022-01-14', '2022-01-15', '2022-01-16', '2022-01-17',
       '2022-01-18', '2022-01-19', '2022-01-20', '2022-01-21',
       '2022-01-22'], dtype=object)

In [57]:
all_data.to_csv(r'C:\Users\tiffany.ona\Documents\working_memory\code\video_analysis\data\all_data.csv', index=False)

In [3]:
all_data = pd.read_csv(r'C:\Users\tiffany.ona\Documents\working_memory\code\video_analysis\data\all_data.csv')

C:\Users\tiffany.ona\AppData\Local\Temp\ipykernel_1892\4077238177.py:1: DtypeWarning: Columns (19,20,21,22) have mixed types. Specify dtype option on import or set low_memory=False.
  all_data = pd.read_csv(r'C:\Users\tiffany.ona\Documents\working_memory\code\video_analysis\data\all_data.csv')
